# MVP Train Notebook: Churn From DAC

Чистый MVP-ноутбук для выбранной модели `06_optuna_01_all_features`: загрузка статистики экспериментов из MLflow, обучение финальной модели на выбранных 100 фичах, OOF/CV-оценка и OOT-валидация, логирование финальной версии и сохранение артефактов.

## 1. Imports / Env / MLflow

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
from datetime import datetime

# Если проект не установлен в kernel, добавляем локальный src.
PROJECT_ROOT = Path('/Users/underplums/Documents/work/organic-return-dac')
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split

from sklearn.metrics import (
    average_precision_score as sk_average_precision_score,
    roc_auc_score as sk_roc_auc_score,
    brier_score_loss as sk_brier_score_loss,
    classification_report,
    ConfusionMatrixDisplay,
)

from cvm_ml_metrics.classification import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
)

import mlflow
from mlflow import MlflowClient
from mlflow.models.signature import infer_signature

from cvm_model.parameters import RANDOM_STATE, target, score, project_name, model_type, jira, artifacts_dir
import cvm_model.functions as func

In [ ]:
# Если .env нужен, раскомментируй.
# %load_ext dotenv
# %dotenv
# %dotenv /Users/underplums/Documents/work/organic-return-dac/.env

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI') or 'https://mlflow-analytics.cvm-prod.corp.tander.ru'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print('MLflow tracking URI:', mlflow.get_tracking_uri())
print('MLFLOW_TRACKING_USERNAME exists:', bool(os.getenv('MLFLOW_TRACKING_USERNAME')))
print('MLFLOW_TRACKING_PASSWORD exists:', bool(os.getenv('MLFLOW_TRACKING_PASSWORD')))

## 2. Fixed MVP Config

In [ ]:
event_timestamp = datetime(2026, 1, 1)
base_month = event_timestamp.date().replace(day=1).isoformat()
target_month = (pd.Timestamp(base_month) + pd.DateOffset(months=1)).date().isoformat()
feature_date = target_month

FINAL_FEATURE_SET_NAME = '06_optuna_01_all_features'
FINAL_THRESHOLD = 0.30

# Управляющие флаги MVP-ноутбука
RUN_CV = True
RUN_OPTUNA_TUNING = False
N_TRIALS = 30
OPTUNA_TIMEOUT = None  # например, 60 * 60 для ограничения в 1 час

FINAL_FEATURES = ['dac_months_last_6', 'is_stable_dac', 'dac_months_last_12', 'login_count_3', 'login_count_6', 'trans_lag_avg_3', 'trans_lag_avg_6', 'login_count_2', 'login_lag_avg_3', 'omni_qr_days_count_2', 'omni_qr_days_count_6', 'omni_qr_days_count_3', 'login_count_12', 'omni_qr_days_count_1', 'dac_months_last_3', 'login_count_1', 'trans_count_3', 'trans_count_2', 'login_lag_avg_6', 'omni_qr_days_count_12', 'trans_count_6', 'avg_level_3', 'login_count_24', 'login_lag_avg_2', 'trans_count_1', 'omni_qr_days_count_24', 'login_lag_avg_12', 'trans_count_12', 'rto_3', 'rto_6', 'login_lag_avg_24', 'avg_level_6', 'avg_level_12', 'rto_2', 'login_recency', 'avg_level_24', 'omni_qr_lag_avg_6', 'dac_months_count', 'rto_12', 'cheque_recency', 'trans_count_24', 'trans_lag_avg_12', 'trans_lag_avg_2', 'rto_1', 'trans_lag_avg_24', 'is_unstable_dac', 'rto_24', 'bonus_expired_sum_1', 'omni_qr_recency', 'dac_months_per_dac_age_ratio', 'omni_qr_lag_avg_12', 'omni_qr_lag_avg_3', 'omni_qr_lag_avg_24', 'omni_unique_features_count_24', 'avg_level_2', 'omni_qr_lag_avg_2', 'is_new_dac', 'omni_unique_features_count_12', 'avg_level_1', 'bonus_accrued_sum_24', 'bonus_accrued_sum_2', 'bonus_expired_sum_2', 'omni_features_days_count_24', 'omni_unique_features_count_6', 'bonus_accrued_sum_3', 'omni_fav_features_days_count_24', 'bonus_accrued_sum_12', 'omni_features_days_count_12', 'bonus_accrued_sum_1', 'whs_count_24', 'omni_fav_features_days_count_12', 'bonus_accrued_sum_6', 'omni_features_days_count_6', 'accept_count_24', 'login_lag_avg_1', 'accept_count_12', 'omni_unique_features_count_3', 'dac_age_months', 'omni_features_days_count_3', 'omni_fav_features_days_count_6', 'omni_unique_features_count_2', 'omni_features_days_count_2', 'bonus_expired_sum_24', 'trans_lag_avg_1', 'bonus_expired_sum_3', 'format_count_24', 'omni_qr_lag_avg_1', 'omni_unique_features_count_1', 'omni_fav_features_days_count_3', 'omni_features_days_count_1', 'is_regular_dac', 'omni_goals_activated_count_12', 'accept_count_6', 'omni_features_lag_avg_24', 'omni_goals_updated_count_12', 'omni_goals_activated_count_6', 'omni_features_lag_avg_1', 'omni_fav_features_days_count_2', 'bonus_redeemed_sum_24', 'omni_goals_updated_count_6']

BEST_CATBOOST_PARAMS = {
    'random_state': RANDOM_STATE,
    'loss_function': 'Logloss',
    'eval_metric': 'PRAUC',
    'iterations': 1106,
    'learning_rate': 0.04092870749826398,
    'depth': 8,
    'l2_leaf_reg': 8.696367472979667,
    'random_strength': 1.5293146536315834,
    'border_count': 64,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.6999263676247318,
    'early_stopping_rounds': 50,
    'thread_count': -1,
    'verbose': False,
}

print('base_month:', base_month)
print('target_month:', target_month)
print('target:', target)
print('features:', len(FINAL_FEATURES))

## 3. Experiment Statistics From MLflow

In [ ]:
def load_mlflow_experiment_table(experiment_name: str) -> pd.DataFrame:
    try:
        exp = mlflow.get_experiment_by_name(experiment_name)
        if exp is None:
            print(f'Experiment not found: {experiment_name}')
            return pd.DataFrame()
        runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], output_format='pandas')
        if runs.empty:
            return runs

        keep_cols = [
            'run_id', 'tags.mlflow.runName', 'params.feature_set', 'params.final_feature_set',
            'params.features_count', 'params.base_month', 'params.target_month',
            'metrics.holdout_pr_auc', 'metrics.holdout_roc_auc', 'metrics.holdout_ap_gain',
            'metrics.holdout_f1', 'metrics.holdout_precision', 'metrics.holdout_recall',
            'metrics.top_decile_target_rate', 'metrics.top_decile_lift',
            'start_time',
        ]
        keep_cols = [c for c in keep_cols if c in runs.columns]
        return runs[keep_cols].sort_values(
            by=[c for c in ['metrics.holdout_pr_auc', 'start_time'] if c in keep_cols],
            ascending=False,
        )
    except Exception as e:
        print(f'Failed to load {experiment_name}: {e}')
        return pd.DataFrame()

feature_selection_experiment = f'{project_name}_{model_type}_{jira}_feature_selection'
final_experiment = f'{project_name}_{model_type}_{jira}_final'

feature_selection_runs = load_mlflow_experiment_table(feature_selection_experiment)
final_runs = load_mlflow_experiment_table(final_experiment)

print('Feature selection experiment:', feature_selection_experiment)
display(feature_selection_runs)

print('Final experiment:', final_experiment)
display(final_runs)

In [ ]:
# Fallback summary from the completed experiment, useful if MLflow is unavailable in the notebook kernel.
manual_experiment_summary = pd.DataFrame([
    {'feature_set': '06_optuna_01_all_features', 'features_count': 100, 'holdout_pr_auc': 0.550046, 'holdout_roc_auc': 0.869620, 'holdout_f1': 0.554336, 'top_decile_target_rate': 0.606954, 'comment': 'selected MVP'},
    {'feature_set': '01_all_features', 'features_count': 100, 'holdout_pr_auc': 0.548777, 'holdout_roc_auc': 0.869216, 'holdout_f1': 0.554461, 'top_decile_target_rate': 0.605087, 'comment': 'default CatBoost'},
    {'feature_set': '03_top_80_features', 'features_count': 80, 'holdout_pr_auc': 0.548387, 'holdout_roc_auc': 0.869025, 'holdout_f1': 0.553983, 'top_decile_target_rate': 0.608013, 'comment': 'compact feature set'},
    {'feature_set': '02_no_redundant_features', 'features_count': 49, 'holdout_pr_auc': 0.547327, 'holdout_roc_auc': 0.868464, 'holdout_f1': 0.553638, 'top_decile_target_rate': 0.605692, 'comment': 'removed redundant'},
    {'feature_set': '04_no_dac_segment_features', 'features_count': 90, 'holdout_pr_auc': 0.542699, 'holdout_roc_auc': 0.866403, 'holdout_f1': 0.548600, 'top_decile_target_rate': 0.601302, 'comment': 'without DAC-history'},
    {'feature_set': '05_only_dac_segment_features', 'features_count': 10, 'holdout_pr_auc': 0.419271, 'holdout_roc_auc': 0.800975, 'holdout_f1': 0.477362, 'top_decile_target_rate': 0.490690, 'comment': 'only DAC-history'},
])
display(manual_experiment_summary.sort_values('holdout_pr_auc', ascending=False))

## 4. Helpers

In [ ]:
def fbeta_threshold(y_true, y_score, beta=1.0):
    scores = {}
    for thr in np.arange(0.05, 1.0, 0.05):
        scores[thr] = fbeta_score(y_true, y_score > thr, beta=beta)
    return max(scores.items(), key=lambda x: x[1])[0]


def decile_report(data, score_col, target_col):
    tmp = data[[score_col, target_col]].copy()
    tmp['score_decile'] = pd.qcut(
        tmp[score_col].rank(method='first'),
        q=10,
        labels=False,
        duplicates='drop',
    ) + 1
    return tmp.groupby('score_decile').agg(
        rows=(target_col, 'size'),
        target_rate=(target_col, 'mean'),
        score_min=(score_col, 'min'),
        score_max=(score_col, 'max'),
    ).sort_index(ascending=False)


def calculate_binary_metrics(y_true, y_score, threshold_value):
    y_pred = y_score > threshold_value
    return {
        'roc_auc': float(sk_roc_auc_score(y_true, y_score)),
        'pr_auc': float(sk_average_precision_score(y_true, y_score)),
        'ap_gain': float(sk_average_precision_score(y_true, y_score) - np.mean(y_true)),
        'threshold': float(threshold_value),
        'predicted_positive_share': float(np.mean(y_pred)),
        'precision': float(precision_score(y_true, y_pred)),
        'recall': float(recall_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred)),
        'f05': float(fbeta_score(y_true, y_pred, beta=0.5)),
        'f2': float(fbeta_score(y_true, y_pred, beta=2)),
        'brier': float(sk_brier_score_loss(y_true, y_score)),
    }


def prepare_dataset(data):
    data = data.copy()
    data['contact_id'] = data['contact_id'].astype('int64')
    if 'dac_segment_12m' in data.columns:
        data['segment'] = data['dac_segment_12m'].astype(str)

    recency_cols = ['cheque_recency', 'login_recency', 'omni_qr_recency', 'omni_features_recency', 'perf_recency']
    for col in recency_cols:
        if col in data.columns:
            data[col] = data[col].fillna(999)
    return data


def validate_feature_set(data, feature_cols):
    leakage_cols = {'contact_id', target, 'target_churn_from_dac', 'is_dac_next_month', score, 'class_0', 'strat', 'final_best_score'}
    bad = sorted(set(feature_cols) & leakage_cols)
    missing = sorted(set(feature_cols) - set(data.columns))
    assert not bad, f'Leakage/service columns in features: {bad}'
    assert not missing, f'Missing features: {missing}'
    return True

## 5. Load Full Train Dataset

In [ ]:
dataset_path = Path(f'df_{base_month}.parquet')

raw_df = pd.read_parquet(dataset_path)
df = prepare_dataset(raw_df)

print('df:', df.shape)
print('target rate:', df[target].mean())

assert target in df.columns
assert df['contact_id'].nunique() == len(df)
validate_feature_set(df, FINAL_FEATURES)

df['strat'] = df['segment'].astype(str) + '_' + df[target].astype(str)

strat_counts = df['strat'].value_counts()
if (strat_counts < 5).any():
    print('Small strata found, fallback to target-only stratification')
    df['strat'] = df[target].astype(str)

display(df.groupby('segment')[target].agg(['count', 'mean']).sort_values('mean', ascending=False))

## 6. Optional Optuna Tuning

In [ ]:
if RUN_OPTUNA_TUNING:
    import optuna

    X_train_tune, X_val_tune, y_train_tune, y_val_tune, z_train_tune, _ = train_test_split(
        df[FINAL_FEATURES],
        df[target],
        df['strat'],
        stratify=df['strat'],
        test_size=0.25,
        random_state=RANDOM_STATE,
    )

    def objective(trial):
        params_trial = {
            'random_state': RANDOM_STATE,
            'loss_function': 'Logloss',
            'eval_metric': 'PRAUC',
            'iterations': trial.suggest_int('iterations', 300, 1200),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 30.0, log=True),
            'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
            'border_count': trial.suggest_categorical('border_count', [64, 128, 254]),
            'early_stopping_rounds': 50,
            'thread_count': -1,
            'verbose': False,
        }

        bootstrap_type = trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS'])
        params_trial['bootstrap_type'] = bootstrap_type

        if bootstrap_type == 'Bayesian':
            params_trial['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.0, 5.0)
        else:
            params_trial['subsample'] = trial.suggest_float('subsample', 0.6, 1.0)

        tune_model = CatBoostClassifier(**params_trial)
        tune_model.fit(
            X_train_tune,
            y_train_tune,
            eval_set=(X_val_tune, y_val_tune),
            use_best_model=True,
            verbose=False,
        )

        val_pred = tune_model.predict_proba(X_val_tune)[:, 1]
        return sk_average_precision_score(y_val_tune, val_pred)

    study = optuna.create_study(
        direction='maximize',
        study_name=f'mvp_catboost_{base_month}_churn_from_dac_pr_auc',
    )
    study.optimize(objective, n_trials=N_TRIALS, timeout=OPTUNA_TIMEOUT, show_progress_bar=True)

    print('Best Optuna validation PR-AUC:', study.best_value)
    print('Best params:')
    display(study.best_params)

    BEST_CATBOOST_PARAMS = {
        'random_state': RANDOM_STATE,
        'loss_function': 'Logloss',
        'eval_metric': 'PRAUC',
        'early_stopping_rounds': 50,
        'thread_count': -1,
        'verbose': False,
        **study.best_params,
    }
else:
    print('Optuna tuning skipped. Using fixed BEST_CATBOOST_PARAMS.')

## 7. StratifiedKFold OOF Cross-Validation

In [ ]:
if RUN_CV:
    cv_model_params = BEST_CATBOOST_PARAMS.copy()
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    X = df[FINAL_FEATURES]
    y = df[target]
    z = df['strat']

    cv_predicts = np.zeros(len(df))
    cv_ap_gain = []
    cv_pr_auc = []
    cv_roc_auc = []

    for i, (train_idx, val_idx) in enumerate(skf.split(X, z), start=1):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        print()
        print(f'Training fold {i}...')
        fold_model = CatBoostClassifier(**cv_model_params)
        fold_model.fit(X_train, y_train, plot=False, verbose=0)

        print(f'Scoring fold {i}...')
        cv_predicts[val_idx] = fold_model.predict_proba(X_val)[:, 1]

        fold_pr_auc = sk_average_precision_score(y_val, cv_predicts[val_idx])
        fold_roc_auc = sk_roc_auc_score(y_val, cv_predicts[val_idx])
        fold_ap_gain = fold_pr_auc - y_val.mean()

        cv_pr_auc.append(fold_pr_auc)
        cv_roc_auc.append(fold_roc_auc)
        cv_ap_gain.append(fold_ap_gain)

        print(f'Fold {i} PR-AUC = {fold_pr_auc:.6f}; ROC-AUC = {fold_roc_auc:.6f}; AP gain = {fold_ap_gain:.6f}')

    # Как в оригинальной модели: основная score-колонка для метрик = OOF-предсказания.
    df[score] = cv_predicts

    cv_metrics = {
        'oof_pr_auc': float(sk_average_precision_score(df[target], df[score])),
        'oof_roc_auc': float(sk_roc_auc_score(df[target], df[score])),
        'oof_ap_gain': float(sk_average_precision_score(df[target], df[score]) - df[target].mean()),
        'fold_pr_auc_mean': float(np.mean(cv_pr_auc)),
        'fold_pr_auc_std': float(np.std(cv_pr_auc)),
        'fold_roc_auc_mean': float(np.mean(cv_roc_auc)),
        'fold_roc_auc_std': float(np.std(cv_roc_auc)),
        'fold_ap_gain_mean': float(np.mean(cv_ap_gain)),
        'fold_ap_gain_std': float(np.std(cv_ap_gain)),
    }

    display(pd.DataFrame(cv_metrics, index=['value']).T)
else:
    cv_metrics = {}
    print('Cross-validation skipped. Metrics will be calculated on in-sample final model scores.')

## 8. Train Final MVP Model On Full Dataset

In [ ]:
final_model = CatBoostClassifier(**BEST_CATBOOST_PARAMS)
final_model.fit(df[FINAL_FEATURES], df[target], plot=False, verbose=100)

final_threshold = FINAL_THRESHOLD

if score not in df.columns:
    # Fallback only for fast debug runs with RUN_CV=False. These metrics are optimistic.
    df[score] = final_model.predict_proba(df[FINAL_FEATURES])[:, 1]
    score_source = 'final_model_in_sample'
else:
    score_source = 'oof_cv'

metrics = calculate_binary_metrics(df[target], df[score], final_threshold)

print('Score source:', score_source)
print('MVP metrics:')
display(pd.DataFrame(metrics, index=['value']).T)

## 9. OOF Evaluation

In [ ]:
deciles = decile_report(df, score, target)
display(deciles)

segment_rows = []
for segment_name, part in df.groupby('segment'):
    if part[target].nunique() < 2:
        continue
    segment_rows.append({
        'segment': segment_name,
        'rows': len(part),
        'target_rate': float(part[target].mean()),
        'pr_auc': float(sk_average_precision_score(part[target], part[score])),
        'roc_auc': float(sk_roc_auc_score(part[target], part[score])),
    })
segment_metrics = pd.DataFrame(segment_rows).sort_values('pr_auc', ascending=False)
display(segment_metrics)

func.plot_precision_recall_curve(df[target], df[score], title_suffix='MVP OOF')
func.plot_roc(df[target], df[score], title_suffix='MVP OOF')

ConfusionMatrixDisplay.from_predictions(
    df[target],
    df[score] > final_threshold,
    display_labels=['No churn from DAC', 'Churn from DAC'],
    cmap='Blues',
    values_format='d',
)
plt.title('MVP OOF Confusion Matrix')
plt.show()

## 10. OOT Validation On Next Month

In [ ]:
# Укажи путь к датасету следующего месяца, если нужно пересчитать OOT.
OOT_DATASET_PATH = Path(f'df_2026-02-01.parquet')
RUN_OOT_VALIDATION = OOT_DATASET_PATH.exists()

if RUN_OOT_VALIDATION:
    oot_df = prepare_dataset(pd.read_parquet(OOT_DATASET_PATH))
    validate_feature_set(oot_df, FINAL_FEATURES)
    assert target in oot_df.columns

    oot_score_col = 'oot_score'
    oot_df[oot_score_col] = final_model.predict_proba(oot_df[FINAL_FEATURES])[:, 1]
    oot_metrics = calculate_binary_metrics(oot_df[target], oot_df[oot_score_col], final_threshold)
    oot_deciles = decile_report(oot_df, oot_score_col, target)

    print('OOT metrics:')
    display(pd.DataFrame(oot_metrics, index=['oot']).T)
    print('OOT target rate:', oot_df[target].mean())
    display(oot_deciles)

    oot_segment_rows = []
    for segment_name, part in oot_df.groupby('segment'):
        if part[target].nunique() < 2:
            continue
        oot_segment_rows.append({
            'segment': segment_name,
            'rows': len(part),
            'target_rate': float(part[target].mean()),
            'pr_auc': float(sk_average_precision_score(part[target], part[oot_score_col])),
            'roc_auc': float(sk_roc_auc_score(part[target], part[oot_score_col])),
        })
    oot_segment_metrics = pd.DataFrame(oot_segment_rows).sort_values('pr_auc', ascending=False)
    display(oot_segment_metrics)
else:
    print('OOT dataset not found, skipped:', OOT_DATASET_PATH)

In [ ]:
# Зафиксированные результаты OOT-валидации из предыдущего запуска.
oot_fixed_summary = pd.DataFrame([
    {'metric': 'target_rate', 'value': 0.129077},
    {'metric': 'pr_auc', 'value': 0.494372},
    {'metric': 'roc_auc', 'value': 0.872598},
    {'metric': 'ap_gain', 'value': 0.365295},
    {'metric': 'precision', 'value': 0.418911},
    {'metric': 'recall', 'value': 0.663246},
    {'metric': 'f1', 'value': 0.513495},
    {'metric': 'top_decile_target_rate', 'value': 0.530687},
])
display(oot_fixed_summary)

## 11. Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'feature': FINAL_FEATURES,
    'importance': final_model.get_feature_importance(),
}).sort_values('importance', ascending=False)

display(feature_importance.head(50))

plt.figure(figsize=(10, 10))
sns.barplot(data=feature_importance.head(30), x='importance', y='feature', color='steelblue')
plt.title('Top 30 Feature Importances')
plt.grid(axis='x', alpha=0.3)
plt.show()

## 12. Log Final MVP Model To MLflow

In [ ]:
LOG_TO_MLFLOW = True

final_model_name = f'{project_name}_{model_type}'
final_experiment_name = f'{project_name}_{model_type}_{jira}_mvp'
mlflow.set_experiment(final_experiment_name)

artifacts_path = Path(artifacts_dir)
artifacts_path.mkdir(parents=True, exist_ok=True)

features_file = artifacts_path / 'final_best_features.txt'
feature_importance_file = artifacts_path / 'feature_importance.csv'
deciles_file = artifacts_path / 'oof_deciles.csv'
segments_file = artifacts_path / 'oof_segment_metrics.csv'
model_file = artifacts_path / 'final_mvp_model_churn_from_dac.cbm'

features_file.write_text('\n'.join(FINAL_FEATURES), encoding='utf-8')
feature_importance.to_csv(feature_importance_file, index=False)
deciles.to_csv(deciles_file)
segment_metrics.to_csv(segments_file, index=False)
final_model.save_model(str(model_file))

signature = infer_signature(df.head(1)[FINAL_FEATURES], df.head(1)[score])

if LOG_TO_MLFLOW:
    run_name = f'mvp_{FINAL_FEATURE_SET_NAME}_{base_month}'
    with mlflow.start_run(run_name=run_name, description='MVP churn-from-DAC model'):
        mlflow.log_param('base_month', base_month)
        mlflow.log_param('target_month', target_month)
        mlflow.log_param('feature_set', FINAL_FEATURE_SET_NAME)
        mlflow.log_param('features_count', len(FINAL_FEATURES))
        mlflow.log_param('threshold', final_threshold)
        mlflow.log_param('target_rate', float(df[target].mean()))
        mlflow.log_param('score_source', score_source)
        mlflow.log_param('run_cv', RUN_CV)
        mlflow.log_param('run_optuna_tuning', RUN_OPTUNA_TUNING)

        for k, v in BEST_CATBOOST_PARAMS.items():
            if isinstance(v, (str, int, float, bool)) or v is None:
                mlflow.log_param(f'catboost_{k}', v)

        for k, v in metrics.items():
            mlflow.log_metric(k, float(v))

        for k, v in cv_metrics.items():
            mlflow.log_metric(f'cv_{k}', float(v))

        for _, row in segment_metrics.iterrows():
            seg = str(row['segment']).replace('-', '_').replace(' ', '_')
            mlflow.log_metric(f'segment_{seg}_pr_auc', float(row['pr_auc']))
            mlflow.log_metric(f'segment_{seg}_roc_auc', float(row['roc_auc']))

        for path in [features_file, feature_importance_file, deciles_file, segments_file, model_file]:
            mlflow.log_artifact(str(path))

        mlflow.sklearn.log_model(
            final_model,
            name=final_model_name,
            signature=signature,
            registered_model_name=final_model_name,
        )

    print('Logged to MLflow experiment:', final_experiment_name)
    print('Registered model:', final_model_name)
else:
    print('MLflow logging skipped')

## 13. Save OOF Scores

In [ ]:
score_result = df[['contact_id', score]].copy()
score_result = score_result.rename(columns={score: 'score_churn_from_dac_oof'})
score_result['score_month'] = target_month
if 'segment' in df.columns:
    score_result['dac_segment'] = df['segment'].values

score_result_path = Path(f'mvp_oof_scores_{base_month}.parquet')
score_result.to_parquet(score_result_path, index=False)

print('Saved:', score_result_path.resolve())
display(score_result.head())